# 📄 D2 — Full Scientific-Paper Pipeline: ingest → store → search → graph

**Deliverable 2 · CSAI415.** Companion to `reports/D2/D2_report.md`, in the style
of the Week 5 (Mongo + Qdrant) and Week 6 (Neo4j) labs: a **single, self-contained
notebook** that runs the whole pipeline **in-memory** — **no Docker required** —
by importing the team's *actual* production modules (`csai415.ingest`,
`csai415.api`, `csai415.qdrant_dense`, `scripts.seed_neo4j`, `csai415.eval`).

Nothing is faked: PDFs are parsed + embedded with the real `ingest.py`; Qdrant
runs in-memory (`QdrantClient(":memory:")`); Mongo via `mongomock`; the real
FastAPI `/search` via `TestClient`; the graph via **Neo4j when configured** (set
it in `.env` — see §1) or an in-notebook `networkx` fallback that reuses the
team's `seed_graph` loader.

| § | Stage | Rubric |
|---|---|---|
| 2 | **Ingestion** — PDF → text → chunks → embeddings → retrieve | Ingest 3% |
| 3–5 | Storage — Mongo (mongomock) + Qdrant (in-memory, 3 collections) | Ingest 3% |
| 6–7 | FastAPI `/search` — healthz + 3 routes, no leakage | Engineering 2% · Hybrid 5% |
| 8 | Metrics — deployed endpoint vs committed CSV | Hybrid 5% |
| 9–10 | Graph build + the 5 documented queries | Graph 5% |
| 11 | **Intermediate graph analytics** — 5 deeper queries | Graph 5% |

Retrieval **quality is inherited** from the D1 BOHB blessed config — D2 moves the
retriever onto the service stack, it does not re-tune it.

## Section 0 — Bootstrap (Colab / fresh environment)

**Run this cell first.** On Google Colab (or any machine without the repo) it
clones `waf-iq/special-topics` and installs the Python dependencies, so every
`from csai415...` import below resolves and `data/`, `configs/` are present. From
inside an existing clone it just installs deps and continues.

> **Neo4j on Colab:** a freshly-cloned repo has no `.env` (it's gitignored), so
> set the credentials directly before §9, e.g. in a cell:
> `import os; os.environ["NEO4J_URL"]="neo4j+s://...."; os.environ["NEO4J_USER"]="neo4j"; os.environ["NEO4J_PASSWORD"]="..."`


In [1]:
# ── Bootstrap: run me FIRST (Colab / any fresh machine) ───────────────────────
import importlib.util, os, subprocess, sys
from pathlib import Path

DEPS = ["qdrant-client", "mongomock", "networkx", "neo4j", "python-dotenv",
        "pymupdf", "sentence-transformers", "rank_bm25", "fastapi", "httpx",
        "pandas", "pyarrow", "arxiv"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS])

def _has_repo(p: Path) -> bool:
    return (p / "src" / "csai415" / "__init__.py").exists()

cwd = Path.cwd()
if not (_has_repo(cwd) or any(_has_repo(d) for d in cwd.parents)):
    # Not inside the repo (e.g. a fresh Colab runtime) -> clone it.
    if not _has_repo(Path("special-topics")):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/waf-iq/special-topics.git"], check=True)
    os.chdir("special-topics")
print("bootstrap done | cwd:", Path.cwd(), "| repo reachable:",
      _has_repo(Path.cwd()) or any(_has_repo(d) for d in Path.cwd().parents))


bootstrap done | cwd: C:\Users\waska\Projects\special-topics\notebooks | repo found: False


## Section 1 — Environment setup

Fresh env / Colab (all in `requirements.txt`):

```bash
pip install qdrant-client mongomock networkx neo4j python-dotenv pymupdf \
            sentence-transformers rank_bm25 fastapi httpx pandas pyarrow
```

**Using a real Neo4j (e.g. Aura):** drop the credentials into a `.env` at the
repo root and §9 picks them up automatically (`load_dotenv()`). Both the repo's
names and Aura's names are accepted:

```ini
# Aura download gives you exactly these:
NEO4J_URI=neo4j+s://xxxxxxxx.databases.neo4j.io
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your-password
```

Without a `.env` (or if it's unreachable) §9 falls back to an in-notebook
`networkx` graph, so the notebook always runs end-to-end.

In [2]:
import os
import sys
import json
import tempfile
from pathlib import Path

import pandas as pd
from IPython.display import display

# Locate the repo root robustly — works whether the kernel's working directory is
# the repo root, notebooks/, or elsewhere (VS Code / JupyterLab set this differently,
# so we don't rely on the folder being named "notebooks").
def _find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "data" / "processed" / "chunks.parquet").exists() or (d / "pyproject.toml").exists() or (d / ".git").exists():
            return d
    return start

ROOT = _find_repo_root(Path.cwd())
os.chdir(ROOT)                              # all later relative paths resolve from here
for extra in ("src", "scripts"):
    if extra not in sys.path:
        sys.path.insert(0, extra)

# Load .env from the repo root by EXPLICIT path. A bare load_dotenv() can miss it in
# notebooks (it searches relative to a script file that does not exist in a kernel).
ENV_PATH = ROOT / ".env"
try:
    from dotenv import load_dotenv
    load_dotenv(ENV_PATH)
    neo = bool(os.environ.get("NEO4J_URL") or os.environ.get("NEO4J_URI"))
    print(f".env at {ENV_PATH}: {'found' if ENV_PATH.exists() else 'MISSING'} | NEO4J creds loaded: {neo}")
except ImportError:
    print("python-dotenv not installed — relying on process env vars only")

CHUNKS_PARQUET = Path("data/processed/chunks.parquet")
print("repo root:", ROOT, "| chunks.parquet present:", CHUNKS_PARQUET.exists())


.env at C:\Users\waska\Projects\special-topics\.env: found | NEO4J creds loaded: True
repo root: C:\Users\waska\Projects\special-topics | chunks.parquet present: True


## Section 2 — Ingestion pipeline (PDF → chunks → embeddings → retrieve) — Ingest (3%)

The front of the pipeline, run live on real arXiv PDFs in `data/raw_pdfs/` using
the team's `ingest.parse_arxiv_pdfs` (PyMuPDF parse + page-aware sliding-window
chunking) and `ingest.embed_chunks` (BGE-small, 384-d). We then build a numpy
`HybridRetriever` over just-ingested chunks and query it — proving
ingest→embed→retrieve end-to-end before we touch the full corpus.

In [3]:
from csai415.ingest import parse_arxiv_pdfs, embed_chunks

local_pdfs = sorted(Path("data/raw_pdfs").glob("*.pdf"))
if local_pdfs:
    sample_pdfs = local_pdfs[:2]
else:
    # Fresh clone / Colab: raw PDFs aren't committed, so fetch 2 cs.CL papers live.
    from csai415.ingest import download_arxiv_demo
    sample_pdfs = download_arxiv_demo(2)
print("ingesting:", [p.name for p in sample_pdfs])

parsed = parse_arxiv_pdfs(sample_pdfs)                 # PDF -> page-aware text chunks
embedded = embed_chunks(parsed)                        # + 384-d BGE embeddings
print(f"\n{len(parsed)} chunks from {len(sample_pdfs)} PDFs | "
      f"embedding dim = {len(embedded['embedding'].iloc[0])}")
display(embedded[["chunk_id", "page_start", "page_end", "text"]].head(3))


ingesting: ['2605.29218v1.pdf', '2605.29224v1.pdf']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (539 > 512). Running this sequence through the model will result in indexing errors


embed_chunks: WARNING — 13/123 chunks exceed 512 tokens and will be truncated during embedding.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


123 chunks from 2 PDFs | embedding dim = 384


,chunk_id,page_start,page_end,text
0,arxiv:2605.29218v1:0,1,1,GTA: Generating Long-Horizon Tasks for Web Age...
1,arxiv:2605.29218v1:1,1,1,"50 websites covering e-commerce, government, f..."
2,arxiv:2605.29218v1:2,1,1,"al., 2025b). This is problematic: prior work h..."


In [4]:
from csai415.retrieve import HybridRetriever, RetrieverConfig

# Tiny retriever over ONLY the freshly-ingested chunks — ingest -> store -> search.
demo_ret = HybridRetriever(embedded.reset_index(drop=True),
                           RetrieverConfig(metric="cosine", candidate_k=10))
demo_q = "what problem does this paper address?"
print(f'query: "{demo_q}"  (over the {len(embedded)} just-ingested chunks)\n')
for cid, score in demo_ret.search_with_scores(demo_q, k=3):
    snippet = embedded.loc[embedded["chunk_id"] == cid, "text"].iloc[0][:90].replace(chr(10), " ")
    print(f"  {score:.3f}  {cid}\n         {snippet}...")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

query: "what problem does this paper address?"  (over the 123 just-ingested chunks)

  0.861  arxiv:2605.29224v1:8
         integer rating mea- suring how closely the content of a retrieved URL pertains to the targ...
  0.838  arxiv:2605.29224v1:37
         Security 25), pages 3827–3844. 11 A Evaluation Protocol and Dataset Construction A.1 Datas...
  0.804  arxiv:2605.29224v1:0
         Relevance as a Vulnerability: How Web Retrieval Degrades Safety Alignment in LLM Agents Wa...


## Section 3 — Load the full corpus

The committed `chunks.parquet`: SciFact (BEIR test split) + 150 arXiv cs.CL
papers + the 5 D1 demo PDFs, each row a chunk with its 384-d BGE embedding and
provenance (`source`, `authors`, `year`, `topic`).

In [5]:
from csai415.retrieve import load_chunks

chunks = load_chunks(CHUNKS_PARQUET)
print(f"total chunks: {len(chunks):,}")
display(chunks.groupby("source").size().rename("chunks").to_frame())

total chunks: 15,760


,chunks
source,
arxiv,9740
arxiv-demo,357
scifact,5663


## Section 4 — MongoDB storage (mongomock) — Ingest & storage (3%)

The lab's `MetadataStore` pattern on `mongomock` (swap `uri=` for real Mongo).
Papers are derived with the team's actual `seed_neo4j.normalize_paper` /
`iter_papers_from_parquet`, so the schema matches the production seed.

In [6]:
import mongomock
from seed_neo4j import iter_papers_from_parquet, normalize_paper

db = mongomock.MongoClient()["csai415"]
papers = [normalize_paper(r) for r in iter_papers_from_parquet(CHUNKS_PARQUET)]
for p in papers:
    p["_id"] = p.pop("paper_id")
db.papers.insert_many(papers)
db.papers.create_index("authors")
db.papers.create_index("year")

print(f"Mongo papers: {db.papers.count_documents({}):,}")
display(pd.DataFrame(list(db.papers.aggregate(
    [{"$group": {"_id": "$source", "n": {"$sum": 1}}}, {"$sort": {"n": -1}}]))))
print("sample:", {k: (v[:3] if isinstance(v, list) else v)
                  for k, v in db.papers.find_one({"source": "arxiv"}).items()})

Mongo papers: 5,338


,n,_id
0,5183,scifact
1,155,arxiv


sample: {'title': 'ATLAS: Agentic or Latent Visual Reasoning? One Word is Enough for Both', 'year': 2026, 'source': 'arxiv', 'authors': ['Ziyu Guo', 'Rain Liu', 'Xinyan Chen'], 'topics': ['cs.CL'], '_id': '2605.15198v1'}


## Section 5 — Qdrant vector store (in-memory) — Ingest & storage (3%)

`QdrantClient(":memory:")` seeded with the same three collections the live stack
carries (full / scifact / arxiv) via the team's `seed_collection_from_parquet`.
Per-source collections keep SciFact eval honest (D2 Risk #4).

In [7]:
from qdrant_client import QdrantClient
from csai415.qdrant_dense import seed_collection_from_parquet
from csai415.api import SOURCE_COLLECTIONS

tmp = Path(tempfile.mkdtemp(prefix="d2nb_"))
subsets = {
    None:      chunks.reset_index(drop=True),
    "scifact": chunks[chunks["source"] == "scifact"].reset_index(drop=True),
    "arxiv":   chunks[chunks["source"].isin({"arxiv", "arxiv-demo"})].reset_index(drop=True),
}
paths = {}
for key, sub in subsets.items():
    p = tmp / f"{key or 'full'}.parquet"; sub.to_parquet(p); paths[key] = p

qdrant = QdrantClient(":memory:")
qcounts = {SOURCE_COLLECTIONS[k]: seed_collection_from_parquet(qdrant, p, collection=SOURCE_COLLECTIONS[k])
           for k, p in paths.items()}
display(pd.DataFrame(qcounts.items(), columns=["collection", "points"]))

,collection,points
0,chunks_bge384,15760
1,chunks_bge384_scifact,5663
2,chunks_bge384_arxiv,10097


## Section 6 — FastAPI `/search` (the real app) — Engineering (2%)

`csai415.api.create_app` with the in-memory Qdrant client injected, driven via
Starlette's `TestClient` (full ASGI/HTTP stack, no uvicorn/Docker). Entering the
client runs the lifespan hook that builds one retriever per source. `GET /healthz`
is 200 only once retrievers load **and** all three Qdrant collections are reachable.

In [8]:
from fastapi.testclient import TestClient
from csai415.api import create_app

api = TestClient(create_app(chunks_path=paths[None], qdrant_client=qdrant))
api.__enter__()                              # run startup lifespan
hz = api.get("/healthz")
print(f"GET /healthz -> {hz.status_code} {hz.json()}")
assert hz.status_code == 200

GET /healthz -> 200 {'status': 'ok'}


## Section 7 — End-to-end `/search` demo — Hybrid retrieval (5%)

The real endpoint on all three routes. Blessed BOHB config is server-side; the
request carries only `query`, `k`, `source`. Asserts confirm per-source routing.

In [9]:
def search(query, k=5, source=None):
    r = api.post("/search", json={"query": query, "k": k, "source": source})
    r.raise_for_status()
    return r.json()

q = "language model pretraining"
for src in ("scifact", "arxiv", None):
    hits = search(q, 5, src)
    print(f'\nquery="{q}"  source={src!r}  -> {len(hits)} hits')
    display(pd.DataFrame(hits)[["chunk_id", "title", "page_range", "score"]])

sci, arx = search(q, 5, "scifact"), search(q, 5, "arxiv")
assert all(h["chunk_id"].startswith("scifact:") for h in sci), "SciFact route leaked"
assert all(h["chunk_id"].startswith("arxiv:")   for h in arx), "arXiv route leaked"
print("\nPASS — per-source routing holds; no cross-corpus contamination.")


query="language model pretraining"  source='scifact'  -> 5 hits


,chunk_id,title,page_range,score
0,scifact:60206680:0,R: A Language for Data Analysis and Graphics,None,0.867019
1,scifact:11943989:0,Baby hands that move to the rhythm of language...,None,0.855137
2,scifact:3095620:1,Distinct Parietal and Temporal Pathways to the...,None,0.831801
3,scifact:18379855:0,The Natural Statistics of Audiovisual Speech,None,0.776591
4,scifact:83707680:0,A forkhead-domain gene is mutated in a severe ...,None,0.748159



query="language model pretraining"  source='arxiv'  -> 5 hits


,chunk_id,title,page_range,score
0,arxiv:2605.31164v1:40,D$^3$: Dynamic Directional Graph-Constrained D...,11-12,0.893977
1,arxiv:2605.31494v1:29,Consolidating Rewarded Perturbations for LLM P...,10-11,0.886861
2,arxiv:2605.30348v1:39,LLMSurgeon: Diagnosing Data Mixture of Large L...,12-13,0.874511
3,arxiv:2605.30717v1:32,Neuron-Level Interventions for Gendered and Ge...,11-12,0.827756
4,arxiv:2605.30348v1:38,LLMSurgeon: Diagnosing Data Mixture of Large L...,11-13,0.794080



query="language model pretraining"  source=None  -> 5 hits


,chunk_id,title,page_range,score
0,arxiv:2605.31164v1:40,D$^3$: Dynamic Directional Graph-Constrained D...,11-12,0.892261
1,arxiv:2605.31494v1:29,Consolidating Rewarded Perturbations for LLM P...,10-11,0.887545
2,arxiv:2605.30348v1:39,LLMSurgeon: Diagnosing Data Mixture of Large L...,12-13,0.876683
3,arxiv:2605.30717v1:32,Neuron-Level Interventions for Gendered and Ge...,11-12,0.836311
4,arxiv:2605.30348v1:38,LLMSurgeon: Diagnosing Data Mixture of Large L...,11-13,0.806717



PASS — per-source routing holds; no cross-corpus contamination.


## Section 8 — Metrics: deployed endpoint vs committed CSV — Hybrid retrieval (5%)

Evaluate the live `/search` (SciFact route) on the same 60-query holdout the D1
run card used, via the shared `csai415.eval.evaluate`. Reproduces
`reports/D2/d2_search_metrics.csv` → `hybrid_blessed` (small NDCG drift = the
cosine-ANN-vs-numpy fusion gap, documented < 0.5pp).

In [10]:
from csai415.eval import evaluate

qa = [json.loads(line) for line in Path("data/gold/qa.jsonl").read_text(encoding="utf-8").splitlines()]
split = json.loads(Path("configs/d1_split_indices.json").read_text(encoding="utf-8"))
holdout = [qa[i] for i in split["holdout"]]

print(f"Evaluating deployed /search on {len(holdout)} holdout queries...")
live = evaluate(lambda q, k, hw: [h["chunk_id"] for h in search(q, k, "scifact")], holdout, k=5)
csv = pd.read_csv("reports/D2/d2_search_metrics.csv").set_index("config").loc["hybrid_blessed"]
print(f"  deployed   NDCG@5={live['ndcg5']:.4f}  Recall@5={live['recall5']:.4f}")
print(f"  committed  NDCG@5={csv['ndcg@5']:.4f}  Recall@5={csv['recall@5']:.4f}")
assert abs(live["recall5"] - csv["recall@5"]) < 0.02
print("PASS — deployed endpoint reproduces the committed hybrid_blessed metrics.")

Evaluating deployed /search on 60 holdout queries...


  deployed   NDCG@5=0.5589  Recall@5=0.6489
  committed  NDCG@5=0.5611  Recall@5=0.6489
PASS — deployed endpoint reproduces the committed hybrid_blessed metrics.


## Section 9 — Graph backend: Neo4j (if configured) or networkx — Graph build (5%)

If `.env` provides a reachable Neo4j (`NEO4J_URI`/`NEO4J_URL` + `NEO4J_USERNAME`/
`NEO4J_USER` + `NEO4J_PASSWORD`), we MERGE the graph with the team's
`seed_neo4j.seed_graph` and run **real Cypher**. Otherwise we build the *same*
graph in-notebook with `networkx`, reusing `seed_graph` against a tiny sink. The
dispatcher `graph_query(cypher, nx_fn, columns)` runs whichever backend is active.

In [11]:
import networkx as nx
from neo4j import GraphDatabase
from seed_neo4j import iter_papers_from_parquet, seed_graph

NEO4J_URL  = os.environ.get("NEO4J_URL")  or os.environ.get("NEO4J_URI")
NEO4J_USER = os.environ.get("NEO4J_USER") or os.environ.get("NEO4J_USERNAME", "neo4j")
NEO4J_PASS = os.environ.get("NEO4J_PASSWORD") or os.environ.get("NEO4J_PASS")

use_neo4j = False
_driver = _session = GRAPH = None
if NEO4J_URL:
    try:
        _driver = GraphDatabase.driver(NEO4J_URL, auth=(NEO4J_USER, NEO4J_PASS) if NEO4J_PASS else None)
        _driver.verify_connectivity()
        use_neo4j = True
        print("Connected to Neo4j — running real Cypher.")
    except Exception as e:
        print(f"NEO4J set but unreachable ({type(e).__name__}: {e}) — networkx fallback.")
else:
    print("No NEO4J_URI/URL in env — using in-notebook networkx graph (set it in .env for real Cypher).")

if use_neo4j:
    _session = _driver.session()
    seed_graph(_session, iter_papers_from_parquet(CHUNKS_PARQUET))   # idempotent MERGE
    n = lambda lbl: _session.run(f"MATCH (x:{lbl}) RETURN count(x) AS n").single()["n"]
    print(f"Neo4j graph: {n('Paper')} Papers, {n('Author')} Authors, {n('Topic')} Topics")
else:
    class _NxSink:
        def __init__(self): self.G = nx.DiGraph()
        def run(self, _stmt, **p):
            if "paper_id" not in p: return None          # constraint statement
            pid = p["paper_id"]
            self.G.add_node(("Paper", pid), kind="Paper", title=p["title"], year=p["year"], source=p["source"])
            for a in p["authors"]:
                self.G.add_node(("Author", a), kind="Author"); self.G.add_edge(("Author", a), ("Paper", pid), rel="WROTE")
            for t in p["topics"]:
                self.G.add_node(("Topic", t), kind="Topic"); self.G.add_edge(("Paper", pid), ("Topic", t), rel="ABOUT")
            return None
    _sink = _NxSink()
    gstats = seed_graph(_sink, iter_papers_from_parquet(CHUNKS_PARQUET), apply_constraints=False)
    GRAPH = _sink.G
    kinds = lambda k: [x for x, d in GRAPH.nodes(data=True) if d["kind"] == k]
    print(f"networkx graph: {len(kinds('Paper'))} Papers, {len(kinds('Author'))} Authors, "
          f"{len(kinds('Topic'))} Topics (loaded={gstats['papers_loaded']}, skipped={gstats['papers_skipped']})")

# Demo author: prefer the one the committed Cypher used, else the busiest.
if use_neo4j:
    DEMO_AUTHOR_NAME = _session.run(
        "MATCH (a:Author) WHERE a.name='Jun Wang' RETURN a.name AS n").single()
    DEMO_AUTHOR_NAME = "Jun Wang" if DEMO_AUTHOR_NAME else _session.run(
        "MATCH (a:Author)-[:WROTE]->(p) RETURN a.name AS n ORDER BY count(p) DESC LIMIT 1").single()["n"]
else:
    if ("Author", "Jun Wang") in GRAPH:
        DEMO_AUTHOR_NAME = "Jun Wang"
    else:
        DEMO_AUTHOR_NAME = max((a for a in kinds("Author")), key=lambda a: GRAPH.out_degree(a))[1]
DEMO_AUTHOR = ("Author", DEMO_AUTHOR_NAME)
print("demo author:", DEMO_AUTHOR_NAME)


def graph_query(cypher, nx_fn, columns, **params):
    # Run a query on whichever backend is active; return a DataFrame.
    rows = [r.data() for r in _session.run(cypher, **params)] if use_neo4j else nx_fn(GRAPH)
    return pd.DataFrame(rows, columns=columns)

Connected to Neo4j — running real Cypher.


Neo4j graph: 155 Papers, 764 Authors, 13 Topics
demo author: Jun Wang


In [12]:
# networkx implementations mirroring each Cypher query (used in fallback mode).
from collections import Counter

def _papers_of(G, a):    return [p for p in G.successors(a) if G.edges[a, p]["rel"] == "WROTE"]
def _authors_of(G, p):   return [a for a in G.predecessors(p) if G.edges[a, p]["rel"] == "WROTE"]
def _topics_of(G, p):    return [t for t in G.successors(p) if G.edges[p, t]["rel"] == "ABOUT"]

# --- core 5 ---
def q1(G):
    return sorted(({"paper_id": pid, "title": G.nodes[("Paper", pid)]["title"], "year": G.nodes[("Paper", pid)]["year"]}
                   for (_, pid) in _papers_of(G, DEMO_AUTHOR)), key=lambda r: (r["year"] or 0), reverse=True)
def q2(G):
    c = Counter(a[1] for p in _papers_of(G, DEMO_AUTHOR) for a in _authors_of(G, p) if a != DEMO_AUTHOR)
    return [{"coauthor": k, "shared_papers": v} for k, v in c.most_common(10)]
def q3(G):
    c = Counter(t[1] for n, d in G.nodes(data=True) if d["kind"] == "Paper"
                and d["year"] is not None and 2025 <= d["year"] <= 2026 for t in _topics_of(G, n))
    return [{"topic": k, "paper_count": v} for k, v in c.most_common(10)]
def q4(G):
    tnode = ("Topic", "cs.CL"); rows = []
    if tnode in G:
        for p in [x for x in G.predecessors(tnode) if G.edges[x, tnode]["rel"] == "ABOUT"]:
            for a in _authors_of(G, p):
                rows.append({"paper_title": G.nodes[p]["title"], "year": G.nodes[p]["year"], "author": a[1]})
    return sorted(rows, key=lambda r: (-(r["year"] or 0), r["paper_title"]))[:10]
def q5(G):
    def auth(topic):
        tn = ("Topic", topic); out = set()
        if tn in G:
            for p in [x for x in G.predecessors(tn) if G.edges[x, tn]["rel"] == "ABOUT"]:
                out |= set(_authors_of(G, p))
        return out
    return [{"author": a[1], "cl_papers": 1, "lg_papers": 1} for a in (auth("cs.CL") & auth("cs.CV"))][:20]

# --- intermediate 5 ---
def cx_prolific(G):
    rows = []
    for a in (n for n, d in G.nodes(data=True) if d["kind"] == "Author"):
        ps = _papers_of(G, a)
        ts = {t[1] for p in ps for t in _topics_of(G, p)}
        rows.append({"author": a[1], "papers": len(ps), "topics": len(ts)})
    return sorted(rows, key=lambda r: (-r["papers"], r["author"]))[:10]
def cx_coauthor_pairs(G):
    c = Counter()
    for p in (n for n, d in G.nodes(data=True) if d["kind"] == "Paper"):
        names = sorted(a[1] for a in _authors_of(G, p))
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                c[(names[i], names[j])] += 1
    return [{"author_a": a, "author_b": b, "shared_papers": v} for (a, b), v in c.most_common(10)]
def cx_interdisciplinary(G):
    rows = []
    for a in (n for n, d in G.nodes(data=True) if d["kind"] == "Author"):
        ts = sorted({t[1] for p in _papers_of(G, a) for t in _topics_of(G, p)})
        if len(ts) >= 2:
            rows.append({"author": a[1], "topic_count": len(ts), "topics": ts})
    return sorted(rows, key=lambda r: (-r["topic_count"], r["author"]))[:10]
def cx_topic_cooccurrence(G):
    c = Counter()
    for p in (n for n, d in G.nodes(data=True) if d["kind"] == "Paper"):
        ts = sorted(t[1] for t in _topics_of(G, p))
        for i in range(len(ts)):
            for j in range(i + 1, len(ts)):
                c[(ts[i], ts[j])] += 1
    return [{"topic_a": a, "topic_b": b, "papers": v} for (a, b), v in c.most_common(10)]
def cx_collaborators(G):
    rows = []
    for a in (n for n, d in G.nodes(data=True) if d["kind"] == "Author"):
        cos = {x for p in _papers_of(G, a) for x in _authors_of(G, p) if x != a}
        if cos:
            rows.append({"author": a[1], "collaborators": len(cos)})
    return sorted(rows, key=lambda r: (-r["collaborators"], r["author"]))[:10]
print("query library ready")

query library ready


## Section 10 — The 5 documented queries — Graph build (5%)

The five queries from `cypher/01..05_*.cypher` (captured in
`reports/D2/d2_cypher_examples.md`), run through the dispatcher: real Cypher on
Neo4j, equivalent traversal on networkx.

In [13]:
from capture_cypher_outputs import QUERIES          # the exact committed Cypher set
core = list(QUERIES.values())
specs = [
    ("01 — papers by " + DEMO_AUTHOR_NAME,            core[0], q1, ["paper_id", "title", "year"]),
    ("02 — top co-authors of " + DEMO_AUTHOR_NAME,    core[1], q2, ["coauthor", "shared_papers"]),
    ("03 — top topics 2025-2026",                     core[2], q3, ["topic", "paper_count"]),
    ("04 — papers + authors about cs.CL",             core[3], q4, ["paper_title", "year", "author"]),
    ("05 — authors on both cs.CL and cs.CV",          core[4], q5, ["author", "cl_papers", "lg_papers"]),
]
for label, qdef, nx_fn, cols in specs:
    print(f"\n--- {label} ---")
    display(graph_query(qdef["cypher"], nx_fn, cols))


--- 01 — papers by Jun Wang ---


,paper_id,title,year
0,2605.31196v1,Probing Collision Grounding in Vision-Language...,2026
1,2605.30947v1,Extending AI for Research to the Humanities: A...,2026



--- 02 — top co-authors of Jun Wang ---


,coauthor,shared_papers
0,Xiaohao Xu,1
1,Xiaonan Huang,1
2,Yating Pan,1
3,Jiajun Zhang,1
4,Qi Su,1



--- 03 — top topics 2025-2026 ---


,topic,paper_count
0,cs.CL,113
1,cs.LG,10
2,cs.CV,10
3,cs.CR,5
4,cs.AI,4
5,cs.SE,3
6,cs.IR,3
7,cs.CY,2
8,cs.GT,1
9,cs.MM,1



--- 04 — papers + authors about cs.CL ---


,paper_title,year,author
0,"""Intelegi Româneşte?'' A Recipe for Romanian V...",2026,Mihai Masala
1,"""Intelegi Româneşte?'' A Recipe for Romanian V...",2026,Marius Leordeanu
2,"""Intelegi Româneşte?'' A Recipe for Romanian V...",2026,Mihai Dascalu
3,"""Intelegi Româneşte?'' A Recipe for Romanian V...",2026,Traian Rebedea
4,A Visually Impaired Assistance Benchmark for V...,2026,Yi Zhao
5,A Visually Impaired Assistance Benchmark for V...,2026,Siqi Wang
6,A Visually Impaired Assistance Benchmark for V...,2026,Zhe Hu
7,A Visually Impaired Assistance Benchmark for V...,2026,Yushi Li
8,A Visually Impaired Assistance Benchmark for V...,2026,Jing Li
9,AI for Monitoring and Classifying Data Used in...,2026,Rafael Macalaba



--- 05 — authors on both cs.CL and cs.CV ---


,author,cl_papers,lg_papers
0,Jun Wang,1,1


## Section 11 — Intermediate graph analytics — Graph build (5%)

Deeper questions over the same `(:Author)-[:WROTE]->(:Paper)-[:ABOUT]->(:Topic)`
schema (no `CITES` — deferred to D3): (A) most prolific authors, (B) top
co-authorship pairs, (C) interdisciplinary authors spanning ≥2 topics, (D) topic
co-occurrence, (E) most collaborative authors by distinct co-author count. Each
runs as real Cypher on Neo4j and as a networkx traversal in the fallback.

> **Note on topic co-occurrence (D):** the `networkx` fallback builds from the
> parquet's *primary* `topic` (one per paper), so co-occurrence is empty there;
> the live Neo4j graph is seeded from Mongo's *full* arXiv category list, so it
> returns real topic pairs — a concrete reason to point `.env` at a real Neo4j.

In [14]:
CY_PROLIFIC = '''
MATCH (a:Author)-[:WROTE]->(p:Paper)
OPTIONAL MATCH (p)-[:ABOUT]->(t:Topic)
RETURN a.name AS author, count(DISTINCT p) AS papers, count(DISTINCT t) AS topics
ORDER BY papers DESC, author LIMIT 10
'''
CY_PAIRS = '''
MATCH (a1:Author)-[:WROTE]->(p:Paper)<-[:WROTE]-(a2:Author)
WHERE a1.name < a2.name
RETURN a1.name AS author_a, a2.name AS author_b, count(DISTINCT p) AS shared_papers
ORDER BY shared_papers DESC, author_a LIMIT 10
'''
CY_INTERDISC = '''
MATCH (a:Author)-[:WROTE]->(:Paper)-[:ABOUT]->(t:Topic)
WITH a, count(DISTINCT t) AS topic_count, collect(DISTINCT t.name) AS topics
WHERE topic_count >= 2
RETURN a.name AS author, topic_count, topics
ORDER BY topic_count DESC, author LIMIT 10
'''
CY_COOCCUR = '''
MATCH (t1:Topic)<-[:ABOUT]-(p:Paper)-[:ABOUT]->(t2:Topic)
WHERE t1.name < t2.name
RETURN t1.name AS topic_a, t2.name AS topic_b, count(DISTINCT p) AS papers
ORDER BY papers DESC, topic_a LIMIT 10
'''
CY_COLLAB = '''
MATCH (a:Author)-[:WROTE]->(:Paper)<-[:WROTE]-(co:Author)
WHERE a <> co
RETURN a.name AS author, count(DISTINCT co) AS collaborators
ORDER BY collaborators DESC, author LIMIT 10
'''

print("--- A. Most prolific authors (papers + distinct topics) ---")
display(graph_query(CY_PROLIFIC, cx_prolific, ["author", "papers", "topics"]))

print("--- B. Top co-authorship pairs ---")
display(graph_query(CY_PAIRS, cx_coauthor_pairs, ["author_a", "author_b", "shared_papers"]))

print("--- C. Interdisciplinary authors (>= 2 topics) ---")
display(graph_query(CY_INTERDISC, cx_interdisciplinary, ["author", "topic_count", "topics"]))

print("--- D. Topic co-occurrence pairs (rich only on real Neo4j) ---")
display(graph_query(CY_COOCCUR, cx_topic_cooccurrence, ["topic_a", "topic_b", "papers"]))

print("--- E. Most collaborative authors (distinct co-author count) ---")
display(graph_query(CY_COLLAB, cx_collaborators, ["author", "collaborators"]))

--- A. Most prolific authors (papers + distinct topics) ---


,author,papers,topics
0,Chongrui Ye,2,1
1,Denny Vrandečić,2,1
2,Elena Simperl,2,1
3,Elizabeth Black,2,1
4,Evgeny Kotelnikov,2,2
5,Ge Liu,2,1
6,Gerrit Quaremba,2,1
7,Guang Zhang,2,1
8,Haozhen Zhang,2,1
9,Hongyu Lin,2,1


--- B. Top co-authorship pairs ---


,author_a,author_b,shared_papers
0,Chongrui Ye,Jingjun Xu,2
1,Chongrui Ye,Haozhen Zhang,2
2,Chongrui Ye,Jiaxuan You,2
3,Chongrui Ye,Xueqiang Xu,2
4,Chongrui Ye,Tianyang Luo,2
5,Chongrui Ye,Ge Liu,2
6,Chongrui Ye,Tao Feng,2
7,Denny Vrandečić,Elizabeth Black,2
8,Denny Vrandečić,Elena Simperl,2
9,Denny Vrandečić,Gerrit Quaremba,2


--- C. Interdisciplinary authors (>= 2 topics) ---


,author,topic_count,topics
0,Evgeny Kotelnikov,2,"[cs.CL, cs.SE]"
1,Jun Wang,2,"[cs.CL, cs.CV]"
2,Koustuv Saha,2,"[cs.CL, cs.HC]"
3,Zhongxiang Dai,2,"[cs.AI, cs.LG]"


--- D. Topic co-occurrence pairs (rich only on real Neo4j) ---


,topic_a,topic_b,papers


--- E. Most collaborative authors (distinct co-author count) ---


,author,collaborators
0,Haoyu Huang,15
1,Hong Ting Tsang,15
2,Jeff Pan,15
3,Jiaxin Bai,15
4,Jiaxuan Xiong,15
5,Lihui Liu,15
6,Tianqing Fang,15
7,Tianshi Zheng,15
8,Yangqiu Song,15
9,Yifei Dong,15


## Section 12 — Wrap-up

Full pipeline, end to end, **no Docker**: real ingestion (PDF→chunks→embeddings,
§2), Mongo + in-memory Qdrant storage (§4–5), the real FastAPI `/search` (§6–7),
deployed-endpoint metrics (§8), and a Neo4j-or-networkx graph with 10 queries
(§10–11). Point `.env` at a real Neo4j (Aura or `docker compose`) to run §9–11 as
live Cypher.

Decisions & pitfalls: `reports/D2/D2_report.md §7` — no `CITES` edges (D3);
cosine-ANN + L2 fusion rescore; name-based author dedup; three Qdrant collections
pending a D3 single-collection refactor; cold-start latency outlier.

In [15]:
api.__exit__(None, None, None)
if use_neo4j:
    _session.close(); _driver.close()
print("done.")

done.
